In [7]:
# ==============================================================
# MSc Project – Biometric Security Baseline + Adversarial Testing
# Author: Stella Williams
# ==============================================================

# --------------------------------------------------------------
# Step 1: Initialise model and device
# --------------------------------------------------------------
# Imports + model + device + loss + sim

import torch
import torch.nn as nn
from facenet_pytorch import InceptionResnetV1
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np
import random, cv2
from PIL import Image
from tqdm import tqdm

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load model
model = InceptionResnetV1(pretrained="vggface2").eval().to(device)

# Loss and similarity metrics
criterion = nn.CosineEmbeddingLoss()
cos = nn.CosineSimilarity(dim=1)


Using device: cpu


In [8]:
# --------------------------------------------------------------
# Step 2: Load and Filter LFW Dataset
# --------------------------------------------------------------

lfw_path = "/Users/stel/Documents/Dissertation/msc-biometric-security/Datasets/lfw-dataset"

transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor()
])

lfw_full = datasets.ImageFolder(root=lfw_path, transform=transform)
target_classes = random.sample(lfw_full.classes, 5)
print("Selected identities:", target_classes)

target_idx = [i for i, (_, label) in enumerate(lfw_full)
              if lfw_full.classes[label] in target_classes]

lfw_subset = Subset(lfw_full, target_idx)
lfw_loader = DataLoader(lfw_subset, batch_size=1, shuffle=True)


Selected identities: ['Ontario_Lett', 'Tomas_Enge', 'Andreas_Vinciguerra', 'Ashanti', 'Vidar_Helgesen']


In [9]:
# --------------------------------------------------------------
# Step 3: Attack and Defence Functions
# --------------------------------------------------------------

def fgsm_attack(image, grad, eps):
    return torch.clamp(image + eps * grad.sign(), 0, 1)

def pgd_attack(model, image, eps=0.2, alpha=0.01, steps=40):
    adv = image.clone().detach().to(device)
    adv.requires_grad = True
    for _ in range(steps):
        output = model(adv)
        loss = criterion(output, model(image).detach(), torch.tensor([-1.0]).to(device))
        model.zero_grad(); loss.backward()
        adv = adv + alpha * adv.grad.sign()
        adv = torch.clamp(image + torch.clamp(adv - image, -eps, eps), 0, 1).detach_()
        adv.requires_grad = True
    return adv

def jpeg_defence(img_tensor, quality=40):
    img_np = (img_tensor.detach().squeeze().permute(1,2,0).cpu().numpy() * 255).astype(np.uint8)
    _, enc = cv2.imencode(".jpg", img_np, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    rec = transforms.ToTensor()(Image.fromarray(cv2.cvtColor(dec, cv2.COLOR_BGR2RGB)))
    return rec.unsqueeze(0).to(device)

def blur_defence(img_tensor, kernel_size=5):
    img_np = (img_tensor.detach().squeeze().permute(1,2,0).cpu().numpy() * 255).astype(np.uint8)
    blurred = cv2.GaussianBlur(img_np, (kernel_size, kernel_size), 0)
    rec = transforms.ToTensor()(Image.fromarray(cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB)))
    return rec.unsqueeze(0).to(device)


In [10]:
# --------------------------------------------------------------
# Step 4: Visualise Utility
# --------------------------------------------------------------

def show(img, title="Image"):
    plt.imshow(img.squeeze().permute(1,2,0).detach().cpu())
    plt.title(title)
    plt.axis("off")


In [ ]:
# --------------------------------------------------------------
# Step 5: PGD Benchmark Loop (With JPEG + Blur)
# --------------------------------------------------------------

epsilons = [0.05, 0.1, 0.15, 0.2, 0.5]
sim_threshold = 0.99  # Attack success
# Mark attack successful if similarity drops below this
recov_threshold = 0.9  # Defence success

print("\n### PGD Benchmark Loop ###")
for eps in epsilons:
    attack_success = 0
    jpeg_recovery = 0
    blur_recovery = 0
    total = 0

    for _ in range(30):
        img, _ = next(iter(lfw_loader))
        img = img.to(device)
        embed_orig = model(img)

        # Generate PGD attack
        adv = pgd_attack(model, img, eps=eps, alpha=0.02, steps=100)
        sim_adv = cos(embed_orig, model(adv)).item()

        # Count successful attacks
        if sim_adv < sim_threshold:
            attack_success += 1

        # Apply JPEG defence
        jpeg = jpeg_defence(adv)
        sim_jpeg = cos(embed_orig, model(jpeg)).item()
        if sim_jpeg > recov_threshold:
            jpeg_recovery += 1

        # Apply Blur defence
        blur = blur_defence(adv)
        sim_blur = cos(embed_orig, model(blur)).item()
        if sim_blur > recov_threshold:
            blur_recovery += 1

        total += 1

    print(f"Epsilon {eps:.2f} | Attack Success: {attack_success/total:.2f} | JPEG Recovery: {jpeg_recovery/total:.2f} | Blur Recovery: {blur_recovery/total:.2f}")



### PGD Benchmark Loop ###
Epsilon 0.05 | Attack Success: 0.00 | JPEG Recovery: 0.93 | Blur Recovery: 0.60
